<a href="https://colab.research.google.com/github/dreamer8848/ML-on-Cloud/blob/main/week_5_filter_based_approach.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
cd /content/drive/MyDrive/Colab Notebooks/ML on Cloud

/content/drive/MyDrive/Colab Notebooks/ML on Cloud


In [2]:
import pandas as pd
from sklearn.feature_selection import mutual_info_classif

# Sample dataset
X = pd.DataFrame({
    'foreign_ip': [1, 1, 1, 0, 0, 0, 1, 0],
    'high_amount': [1, 1, 0, 0, 1, 0, 0, 0]
})
y = [1, 1, 1, 0, 0, 0, 1, 0]

# Compute Information Gain (Mutual Information)
ig_scores = mutual_info_classif(X, y, discrete_features=True, random_state=42)
for col, score in zip(X.columns, ig_scores):
    print(f"Information Gain for {col}: {score:.4f}")

Information Gain for foreign_ip: 0.6931
Information Gain for high_amount: 0.0338


In [4]:
from sklearn.feature_selection import SelectKBest, chi2
import pandas as pd

X = pd.DataFrame({
    'mobile_device': [1, 1, 0, 0, 1, 0, 1, 0],
    'vpn_used': [1, 1, 1, 0, 1, 0, 0, 0]
})
y = [1, 1, 0, 0, 1, 0, 1, 0]

selector = SelectKBest(score_func=chi2, k=1)
X_new = selector.fit_transform(X, y)
print("Top Feature:", X.columns[selector.get_support()][0])

Top Feature: mobile_device


In [6]:
import numpy as np, pandas as pd

def fishers_score(X, y):
    scores = {}
    for col in X.columns:
        m1, m0 = X[col][y == 1].mean(), X[col][y == 0].mean()   # class means
        v1, v0 = X[col][y == 1].var(), X[col][y == 0].var()     # class variances
        scores[col] = ((m1 - m0) ** 2) / (v1 + v0)
    return pd.Series(scores)

X = pd.DataFrame({'amt': [100, 110, 105, 5, 10, 8], 'att': [2, 3, 2, 1, 2, 1]})
y = np.array([1, 1, 1, 0, 0, 0])
print(fishers_score(X, y))

amt    302.35461
att      1.50000
dtype: float64


In [7]:
import pandas as pd, numpy as np

df = pd.DataFrame({'A': [1, 2, np.nan, 4, 5],
                   'B': [np.nan, np.nan, np.nan, 4, np.nan],
                   'C': [1, 2, 3, 4, 5]})

thresh = 0.40                                   # max 40% missing allowed
filtered_df = df.loc[:, df.isnull().mean() < thresh]
print("Retained Columns:", filtered_df.columns.tolist())

Retained Columns: ['A', 'C']


In [8]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
import pandas as pd

X = pd.DataFrame({'x1': [1, 2, 3, 4, 5, 6],
                  'x2': [2, 4, 5, 8, 10, 11],
                  'x3': [9, 1, 8, 2, 7, 3]})
y = [0, 0, 0, 1, 1, 1]

sfs = SequentialFeatureSelector(LogisticRegression(), n_features_to_select=2,
                                direction='forward', cv=3)
sfs.fit(X, y)
print("Forward Selection:", X.columns[sfs.get_support()].tolist())

Forward Selection: ['x1', 'x2']


In [9]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pandas as pd

X = pd.DataFrame({'f1': [10, 20, 30, 40, 50, 60],
                  'f2': [1, 0, 1, 0, 1, 0],
                  'f3': [0.1, 0.1, 0.1, 0.1, 0.1, 0.1]})
y = [0, 0, 0, 1, 1, 1]

rfe = RFE(estimator=LogisticRegression(), n_features_to_select=2)
rfe.fit(X, y)
print("RFE Top Features:", X.columns[rfe.support_].tolist())

RFE Top Features: ['f1', 'f2']


In [10]:
import numpy as np, pandas as pd
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

# ---- practice dataset: 4 features, only the first 3 matter ----
rng = np.random.RandomState(42)
n = 400
X = pd.DataFrame({'amt': rng.normal(size=n),          # strong signal
                  'att': rng.normal(size=n),          # medium signal
                  'age': rng.normal(size=n),          # weak signal
                  'noise': rng.normal(size=n)})       # no signal
logit = 2.0 * X['amt'] + 1.0 * X['att'] + 0.3 * X['age'] - 0.5
y = (rng.uniform(size=n) < 1 / (1 + np.exp(-logit))).astype(int)
print("fraud share:", round(y.mean(), 3))

# ---- 3.1 L1 (Lasso): scale first, then look at which weights hit zero ----
Xs = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)

lasso = Lasso(alpha=0.05).fit(Xs, y)                       # handbook style
print("\nLasso coefficients:\n", pd.Series(lasso.coef_, index=X.columns).round(4))
print("Selected by Lasso:", X.columns[lasso.coef_ != 0].tolist())

logit_l1 = LogisticRegression(penalty='l1', solver='liblinear', C=0.05).fit(Xs, y)   # better for a yes/no target
print("\nL1 logistic coefficients:\n", pd.Series(logit_l1.coef_[0], index=X.columns).round(4))
print("Selected by L1 logistic:", X.columns[logit_l1.coef_[0] != 0].tolist())

# ---- 3.2 Random Forest importance (Mean Decrease in Impurity) ----
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
imp = pd.Series(rf.feature_importances_, index=X.columns)
print("\nRandom Forest importance (impurity):\n", imp.sort_values(ascending=False).round(3))

perm = permutation_importance(rf, X_te, y_te, n_repeats=20, random_state=42)
print("\nPermutation importance (test set):\n",
      pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False).round(3))

fraud share: 0.418

Lasso coefficients:
 amt      0.2280
att      0.0614
age      0.0000
noise   -0.0000
dtype: float64
Selected by Lasso: ['amt', 'att']

L1 logistic coefficients:
 amt      1.1898
att      0.3198
age      0.0000
noise    0.0000
dtype: float64
Selected by L1 logistic: ['amt', 'att']

Random Forest importance (impurity):
 amt      0.513
age      0.179
att      0.178
noise    0.130
dtype: float64

Permutation importance (test set):
 amt      0.158
att      0.018
age     -0.008
noise   -0.023
dtype: float64
